### Save Model to gguf and Serve it with Llama.cpp & Ollama

- https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf

In [3]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
#     !pip install --no-deps --upgrade "torchao>=0.16.0"
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [4]:
# !pip install llama-cpp-python
# !pip install --upgrade triton vllm -q

In [5]:
import gc
import torch
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="A7med-Ame3/Qwen2.5-7B-LiveKit-16bit",
    max_seq_length=2048,
    load_in_4bit=False,
    device_map = "auto"
)

In [ ]:
model.push_to_hub_gguf(
    "A7med-Ame3/Qwen2.5-7B-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)

#     "q8_0"    : "Fast conversion. High resource use, but generally acceptable.",
#     "q4_k_m"  : "Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K",

**Horaaaaaaaaaaaaaaaay**

### Try Running it using Llama.cpp

In [9]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="A7med-Ame3/Qwen2.5-7B-GGUF",
	filename="Qwen2.5-7B-LiveKit-16bit.Q4_K_M.gguf",
)

llama_model_loader: loaded meta data with 29 key-value pairs and 339 tensors from /root/.cache/huggingface/hub/models--A7med-Ame3--Qwen2.5-7B-GGUF/snapshots/515d5136fdd04b2b4d118a67f9955dc48ca39e47/./Qwen2.5-7B-LiveKit-16bit.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                               general.name str              = Unsloth_Gguf_5Dz6X_52
llama_model_loader: - kv   6:  

In [10]:
llm.create_chat_completion(
	messages = "No input example has been defined for this model task."
)

llama_perf_context_print:        load time =    2884.10 ms
llama_perf_context_print: prompt eval time =    2883.68 ms /    24 tokens (  120.15 ms per token,     8.32 tokens per second)
llama_perf_context_print:        eval time =    7301.63 ms /    23 runs   (  317.46 ms per token,     3.15 tokens per second)
llama_perf_context_print:       total time =   10202.63 ms /    47 tokens
llama_perf_context_print:    graphs reused =         22


{'id': 'chatcmpl-c5971086-3e09-41ec-a065-faaea1238226',
 'object': 'chat.completion',
 'created': 1785239379,
 'model': '/root/.cache/huggingface/hub/models--A7med-Ame3--Qwen2.5-7B-GGUF/snapshots/515d5136fdd04b2b4d118a67f9955dc48ca39e47/./Qwen2.5-7B-LiveKit-16bit.Q4_K_M.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'هل فيه فرق بين الحديث الضعيف والحديث الموضوع لما بيتقال إنه حديث نبوي؟'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 24, 'completion_tokens': 23, 'total_tokens': 47}}

In [11]:
output = llm(
    "ما هى ايام الصيام فى الاسلام ؟",
    max_tokens=256,
)

output

llama_perf_context_print:        load time =    2884.10 ms
llama_perf_context_print: prompt eval time =    1279.36 ms /    12 tokens (  106.61 ms per token,     9.38 tokens per second)
llama_perf_context_print:        eval time =   41524.46 ms /   132 runs   (  314.58 ms per token,     3.18 tokens per second)
llama_perf_context_print:       total time =   42909.90 ms /   144 tokens
llama_perf_context_print:    graphs reused =        131


{'id': 'cmpl-ec6e5a0d-8b02-4b96-9ed5-649aaca3bb6f',
 'object': 'text_completion',
 'created': 1785239389,
 'model': '/root/.cache/huggingface/hub/models--A7med-Ame3--Qwen2.5-7B-GGUF/snapshots/515d5136fdd04b2b4d118a67f9955dc48ca39e47/./Qwen2.5-7B-LiveKit-16bit.Q4_K_M.gguf',
 'choices': [{'text': ' - الإسلام سؤال وجواب\nما هى ايام الصيام فى الاسلام ؟\nالصيام هو اجتناب الطعام والشراب والجماع والجماعات المحرمة من الأهل والأصدقاء من طلوع الفجر لحد غروب الشمس، وهو من أهم الأركان الأربعة للإسلام وأوجب الأعمال الصالحة بعد الشهادتين. وأيام الصيام هي الأيام العادية من الشهر الهجري، وبتكون 29 أو 30 يوم حسب القمر، والأيام دي بتصبح أيام عيد بعد ما تطلع شمس العيد.',
   'index': 0,
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 12, 'completion_tokens': 132, 'total_tokens': 144}}

In [12]:
print(f"Shekish Answer: {output['choices'][0]['text']}")
print(f"Input Tokens (Prompt): {output['usage']['prompt_tokens']}")
print(f"Output Tokens (Response): {output['usage']['completion_tokens']}")
print(f"You paid by these tokens -> {output['usage']['total_tokens']}")

Shekish Answer:  - الإسلام سؤال وجواب
ما هى ايام الصيام فى الاسلام ؟
الصيام هو اجتناب الطعام والشراب والجماع والجماعات المحرمة من الأهل والأصدقاء من طلوع الفجر لحد غروب الشمس، وهو من أهم الأركان الأربعة للإسلام وأوجب الأعمال الصالحة بعد الشهادتين. وأيام الصيام هي الأيام العادية من الشهر الهجري، وبتكون 29 أو 30 يوم حسب القمر، والأيام دي بتصبح أيام عيد بعد ما تطلع شمس العيد.
Input Tokens (Prompt): 12
Output Tokens (Response): 132
You paid by these tokens -> 144


### Try Running it using Ollama

In [15]:
!apt-get update -y && apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,849 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.

In [16]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%############                                  55.9%###############################    98.5%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [17]:
!ollama run hf.co/A7med-Ame3/Qwen2.5-7B-GGUF:Q4_K_M

]11;?\Error: could not connect to ollama server, run 'ollama serve' to start it


In [18]:
import subprocess
import time

process = subprocess.Popen(["ollama", "serve"])

time.sleep(5)
print("✅ Ollama Server is up and running!")

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIAb5CZrGWlfWOjBq2GobDphNjJuJ5xULJem8gPPr1pmA



time=2026-07-28T12:01:13.327Z level=INFO source=routes.go:1947 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

✅ Ollama Server is up and running!


time=2026-07-28T12:01:19.891Z level=INFO source=types.go:32 msg="inference compute" id=1 filter_id=1 library=CUDA compute=7.5 name=CUDA1 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:05.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-07-28T12:01:19.891Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.3 GiB"
time=2026-07-28T12:01:19.891Z level=INFO source=routes.go:2054 msg="vram-based default context" total_vram="29.1 GiB" default_num_ctx=32768


In [19]:
!pip install ollama -q

In [21]:
import ollama

print("Downloading model into Ollama...")
ollama.pull("hf.co/A7med-Ame3/Qwen2.5-7B-GGUF:Q4_K_M")
print("Model pulled successfully!")

time=2026-07-28T12:02:50.423Z level=INFO source=download.go:181 msg="downloading f2e538f01e88 in 16 292 MB part(s)"
time=2026-07-28T12:03:48.841Z level=INFO source=download.go:181 msg="downloading e94a8ecb9327 in 1 1.6 KB part(s)"
time=2026-07-28T12:03:49.227Z level=INFO source=download.go:181 msg="downloading e2208325e89e in 1 478 B part(s)"


[GIN] 2026/07/28 - 12:04:04 | 200 |         1m14s |       127.0.0.1 | POST     "/api/pull"
Model pulled successfully!


In [22]:
# !ollama run hf.co/A7med-Ame3/Qwen2.5-7B-GGUF:Q4_K_M "السلام عليكم، عرف نفسك"

response = ollama.chat(
    model="hf.co/A7med-Ame3/Qwen2.5-7B-GGUF:Q4_K_M",
    messages=[
        {"role": "user", "content": "كيف يدخل غير المسلم لدين الاسلام ؟"}
    ]
)

print(response["message"]["content"])

time=2026-07-28T12:04:08.968Z level=INFO source=sched.go:1019 msg="selecting single GPU for llama-server model" main_gpu=0 id=1 filter_id=1 library=CUDA name=CUDA1 description="Tesla T4" integrated=false predicted="6.1 GiB" available="14.5 GiB"
time=2026-07-28T12:04:08.968Z level=INFO source=sched.go:1138 msg="selecting GPU backend for llama-server model" library=CUDA gpu_count=1 available_gpu_count=2
time=2026-07-28T12:04:08.969Z level=INFO source=sched.go:1221 msg="disabling mmap for llama-server load due to host memory pressure" model=/root/.ollama/models/blobs/sha256-f2e538f01e883d45e81ca9c5e8d63e4d0e0b1f183deedfc9d55f83d8746e9457 model_size="4.4 GiB" loaded_mmap_size="0 B" headroom="7.5 GiB" system_free="948.6 MiB" system_total="30.0 GiB" predicted_vram="6.1 GiB" available_vram="14.5 GiB"
time=2026-07-28T12:04:08.969Z level=INFO source=server.go:109 msg="using llama-server for model" model=/root/.ollama/models/blobs/sha256-f2e538f01e883d45e81ca9c5e8d63e4d0e0b1f183deedfc9d55f83d874

[GIN] 2026/07/28 - 12:05:36 | 200 |         1m29s |       127.0.0.1 | POST     "/api/chat"
يدخل الإسلام بثلاثة أركان أساسيين: First, الشهادتين، وهي إقرار بأن لا إله إلا الله وأن محمدًا رسول الله. Second, الإيمان بأربع عشرة صفة عظيمة لله سبحانه وتعالى. وأخيرًا، التسليم بالجبروت والملك والمقدرة. بعدها بيحلف بالله إنه هيكون مسلم ويستمد الإسلام بسنته الصحيحة.


slot print_timing: id  0 | task 0 | prompt eval time =   38598.29 ms /    18 tokens ( 2144.35 ms per token,     0.47 tokens per second)
slot print_timing: id  0 | task 0 |        eval time =   13074.76 ms /    98 tokens (  133.42 ms per token,     7.50 tokens per second)
slot print_timing: id  0 | task 0 |       total time =   51673.05 ms /   116 tokens
slot print_timing: id  0 | task 0 |    graphs reused =         97
slot      release: id  0 | task 0 | stop processing: n_tokens = 115, truncated = 0
srv  update_slots: all slots are idle


In [23]:
print(response["message"]["content"])

يدخل الإسلام بثلاثة أركان أساسيين: First, الشهادتين، وهي إقرار بأن لا إله إلا الله وأن محمدًا رسول الله. Second, الإيمان بأربع عشرة صفة عظيمة لله سبحانه وتعالى. وأخيرًا، التسليم بالجبروت والملك والمقدرة. بعدها بيحلف بالله إنه هيكون مسلم ويستمد الإسلام بسنته الصحيحة.
